In [1]:
%cd ../..

/home/eli/AnacondaProjects/epych


In [2]:
%env DASK_LOGGING__DISTRIBUTED=CRITICAL
%env OMPI_MCA_btl_sm_backing_directory=/mnt/data/tmp_storage
%env SPYTMPDIR=/mnt/data/tmp_storage
%env SPYLOGLEVEL=CRITICAL
%env SPYPARLOGLEVEL=CRITICAL

env: DASK_LOGGING__DISTRIBUTED=CRITICAL
env: OMPI_MCA_btl_sm_backing_directory=/mnt/data/tmp_storage
env: SPYTMPDIR=/mnt/data/tmp_storage
env: SPYLOGLEVEL=CRITICAL
env: SPYPARLOGLEVEL=CRITICAL


In [3]:
import collections
import glob
import functools
from hyppo.ksample import Hotelling
import logging
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import quantities as pq
import scipy.stats as stats

import epych
from epych.statistics import alignment

[striatum:1048511] shmem: mmap: an error occurred while determining whether or not /tmp/ompi.striatum.1000/jf.0/1705312256/shared_mem_cuda_pool.striatum could be created.
[striatum:1048511] create_and_attach: unable to create shared memory BTL coordinating structure :: size 134217728 


In [4]:
%matplotlib inline

In [5]:
logging.basicConfig(level=logging.INFO)

In [6]:
CONDITIONS = ["go_gloexp", "go_seqctl", "lo_gloexp", "lonaive", "lo_rndctl", "igo_seqctl"]
PRETRIAL_SECONDS = 0.5
POSTTRIAL_SECONDS = 0.5
FORCE_RECOMPUTE = True
anatomical_areas = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']

In [7]:
NWB_SUBJECTS = glob.glob('/mnt/data/000253/sub-*/')

In [8]:
PILOT_FILES = []

In [9]:
ODDBALL_ONSET = pq.Quantity(-1.9017372477960602e-14) * pq.second
ODDBALL_OFFSET = pq.Quantity(0.5004545430388676) * pq.second
OFFSET = pq.Quantity(0.020) * pq.second

In [10]:
aligner = epych.statistics.alignment.AlignmentSummary.unpickle("/mnt/data/000253/visual_alignment")
AREA_COUNTER = collections.Counter()

In [11]:
def visual_align(signal):
    visual = signal.select_channels(["VIS" in loc for loc in signal.channels.location]).median_filter()
    area = alignment.location_prefix(None, visual)
    result = aligner.stats[area].align(AREA_COUNTER[area], visual)
    AREA_COUNTER[area] += 1
    return result

In [12]:
def samplings(s, subject_dir, cond):
    subject = subject_dir.split('/')[-2]
    sampling = epych.recording.Sampling.unpickle(subject_dir + "/" + cond).smap(
        lambda sig: visual_align(sig)[(ODDBALL_ONSET - OFFSET).magnitude:(ODDBALL_OFFSET + OFFSET).magnitude]
    )
    logging.info("Loaded LFPs for %s in subject %s" % (cond, subject))
    yield sampling
    del sampling

In [13]:
def initialize_spectrum(key, signal):
    area = os.path.commonprefix([loc for loc in signal.channels.location])
    return epych.statistics.spectrum.PowerSpectrum(signal.df, signal.channels, signal.f0, taper="hann", time_window=0.500)

In [14]:
summaries = {cond: {} for cond in CONDITIONS}

In [15]:
for cond in CONDITIONS:
    global AREA_COUNTER
    AREA_COUNTER = collections.Counter()
    for s, subject_dir in enumerate(sorted(NWB_SUBJECTS)):
        subject = subject_dir.split('/')[-2]
        if not os.path.exists(subject_dir + "/" + cond):
            continue
        if os.path.exists("/mnt/data/000253/%s/spectrum_%s" % (subject_dir, cond)) and not FORCE_RECOMPUTE:
            summaries[cond][subject] = epych.statistic.Summary.unpickle("/mnt/data/000253/%s/spectrum_%s" % (subject_dir, cond),
                                                                        epych.statistics.spectrum.PowerSpectrum)
        else:
            summaries[cond][subject] = epych.statistic.Summary(alignment.location_prefix, initialize_spectrum)
            summaries[cond][subject].calculate(samplings(s, subject_dir, cond))
            summaries[cond][subject].pickle("/mnt/data/000253/%s/spectrum_%s" % (subject_dir, cond))
        logging.info("Analyzed spectra from %s LFPs in subject %s" % (cond, subject))

INFO:root:Loaded LFPs for go_gloexp in subject sub-621890
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:14<00:00,  2.83s/it]
INFO:root:Analyzed spectra from go_gloexp LFPs in subject sub-621890
INFO:root:Loaded LFPs for go_gloexp in subject sub-632485
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:14<00:00,  2.82s/it]
INFO:root:Analyzed spectra from go_gloexp LFPs in

In [16]:
aperiodic_params = {cond: {area: [] for area in anatomical_areas} for cond in CONDITIONS}

for cond in CONDITIONS:
    for s, subject_dir in enumerate(sorted(NWB_SUBJECTS)):
        subject = subject_dir.split('/')[-2]
        for area in summaries[cond][subject].stats:
            if area not in anatomical_areas:
                continue
            aperiodic_params[cond][area].append(summaries[cond][subject].stats[area].aperiodic_parameters(channel_mean=True))
    for area in anatomical_areas:
        aperiodic_params[cond][area] = np.stack(aperiodic_params[cond][area], axis=0).squeeze()

In [17]:
CONTRASTS = [("go_contrast", "go_gloexp", "go_seqctl"), ("ssa", "lo_gloexp", "igo_seqctl"), ("dd", "lo_rndctl", "lonaive")]
contrast_results = {name: {area: None for area in anatomical_areas} for (name, _, _) in CONTRASTS}

In [18]:
for name, condl, condr in CONTRASTS:
    for area in anatomical_areas:
        contrast_results[name][area] = stats.ttest_ind(aperiodic_params[condl][area], aperiodic_params[condr][area], equal_var=False, method=stats.PermutationMethod(n_resamples=1000))

In [19]:
for name, _, _ in CONTRASTS:
    for area in anatomical_areas:
        logging.info("t-test results for %s contrast in area %s: %s" % (name, area, str(contrast_results[name][area])))

INFO:root:t-test results for go_contrast contrast in area VISp: TtestResult(statistic=array([-0.5245, -0.2198]), pvalue=array([0.5814, 0.8092]), df=array([23.0489, 23.8528]))
INFO:root:t-test results for go_contrast contrast in area VISl: TtestResult(statistic=array([-0.0608,  0.0519]), pvalue=array([0.973 , 0.8991]), df=array([23.9315, 23.8458]))
INFO:root:t-test results for go_contrast contrast in area VISrl: TtestResult(statistic=array([-0.1569,  0.008 ]), pvalue=array([0.8771, 0.997 ]), df=array([25.977 , 25.7892]))
INFO:root:t-test results for go_contrast contrast in area VISal: TtestResult(statistic=array([-0.5097, -0.4838]), pvalue=array([0.6194, 0.6194]), df=array([19.8901, 19.8228]))
INFO:root:t-test results for go_contrast contrast in area VISpm: TtestResult(statistic=array([-0.0159,  0.0874]), pvalue=array([0.9371, 0.987 ]), df=array([24.7277, 25.9492]))
INFO:root:t-test results for go_contrast contrast in area VISam: TtestResult(statistic=array([-0.5682, -0.5992]), pvalue=a

In [20]:
fits = {cond: {area: [] for area in anatomical_areas} for cond in CONDITIONS}

for cond in CONDITIONS:
    for s, subject_dir in enumerate(sorted(NWB_SUBJECTS)):
        subject = subject_dir.split('/')[-2]
        for area in summaries[cond][subject].stats:
            if area not in anatomical_areas:
                continue
            fits[cond][area].append(summaries[cond][subject].stats[area].fooofed(channel_mean=True, space="linear", report=True)[-1])

The FOOOF algorithm (version 1.1.0) was used to parameterize neural power spectra. Settings for the algorithm were set as: peak width limits : (0.5, 12.0); max number of peaks : inf; minimum peak height : 0.0; peak threshold : 2.0; and aperiodic mode : 'fixed'. Power spectra were parameterized across the frequency range 2.0 to 150.0 Hz.
The FOOOF algorithm (version 1.1.0) was used to parameterize neural power spectra. Settings for the algorithm were set as: peak width limits : (0.5, 12.0); max number of peaks : inf; minimum peak height : 0.0; peak threshold : 2.0; and aperiodic mode : 'fixed'. Power spectra were parameterized across the frequency range 2.0 to 150.0 Hz.
The FOOOF algorithm (version 1.1.0) was used to parameterize neural power spectra. Settings for the algorithm were set as: peak width limits : (0.5, 12.0); max number of peaks : inf; minimum peak height : 0.0; peak threshold : 2.0; and aperiodic mode : 'fixed'. Power spectra were parameterized across the frequency range 

In [21]:
for area in anatomical_areas:
    nfreqs = []
    for cond in CONDITIONS:
        nfreqs.append([fit.shape[0] for fit in fits[cond][area]])
    nfreqs = np.array(nfreqs).min()
    for cond in CONDITIONS:
        fits[cond][area] = [fit[:nfreqs] for fit in fits[cond][area]]
        fits[cond][area] = np.stack(fits[cond][area], axis=0).squeeze()

In [22]:
CONTRASTS = [("go_contrast", "go_gloexp", "go_seqctl"), ("ssa", "lo_gloexp", "igo_seqctl"), ("dd", "lo_rndctl", "lonaive")]
contrast_results = {name: {area: None for area in anatomical_areas} for (name, _, _) in CONTRASTS}

In [23]:
for name, condl, condr in CONTRASTS:
    for area in anatomical_areas:
        contrast_results[name][area] = stats.ttest_ind(fits[condl][area], fits[condr][area], equal_var=False, method=stats.PermutationMethod(n_resamples=1000))

In [24]:
for name, _, _ in CONTRASTS:
    for area in anatomical_areas:
        logging.info("Permutation-based t-test results for %s contrast in area %s, α=0.05: %s" % (name, area, (contrast_results[name][area].pvalue < 0.05).any()))
        logging.info("Permutation-based t-test results for %s contrast in area %s, α=0.01: %s" % (name, area, (contrast_results[name][area].pvalue < 0.01).any()))

INFO:root:Permutation-based t-test results for go_contrast contrast in area VISp, α=0.05: False
INFO:root:Permutation-based t-test results for go_contrast contrast in area VISp, α=0.01: False
INFO:root:Permutation-based t-test results for go_contrast contrast in area VISl, α=0.05: False
INFO:root:Permutation-based t-test results for go_contrast contrast in area VISl, α=0.01: False
INFO:root:Permutation-based t-test results for go_contrast contrast in area VISrl, α=0.05: False
INFO:root:Permutation-based t-test results for go_contrast contrast in area VISrl, α=0.01: False
INFO:root:Permutation-based t-test results for go_contrast contrast in area VISal, α=0.05: False
INFO:root:Permutation-based t-test results for go_contrast contrast in area VISal, α=0.01: False
INFO:root:Permutation-based t-test results for go_contrast contrast in area VISpm, α=0.05: False
INFO:root:Permutation-based t-test results for go_contrast contrast in area VISpm, α=0.01: False
INFO:root:Permutation-based t-test